# Lesson 2.3 · Run calls, return results, parallel tool calls

**Goal:** handle a reply that asks for several tools at once: run them (at the same time when they only read), and send back exactly one result per call, matched by id.

**What you will be able to do**
- Read a reply whose `tool_calls` list has 4 items, each with its own id.
- Run the calls one by one (3.2 s in the video) or in a thread pool (0.8 s) and get the same results.
- Return one `tool` message per id, and see what happens when one is missing (no error, a wrong diagnosis).
- Write a loop that works for a model that batches (Qwen) and one that doesn't (gpt-oss).

> Three cells **call the API** (about 15 small calls in all, saved in `./cache`, so re-runs are free and work without a key).

In [1]:
# Setup: run this cell first.
# It finds the course's nightwatch/ folder (the Brightcart simulator), loads GROQ_API_KEY from .env if there is one,
# and builds `chat(...)`: a client call with a record/replay cache in ./cache, so re-runs cost nothing.
import hashlib, json, os, sys
from pathlib import Path

HERE = Path.cwd()
NW = next((p / "nightwatch" for p in [HERE, *HERE.parents] if (p / "nightwatch" / "brightcart").is_dir()), None)
assert NW is not None, "Can't find nightwatch/. Open this notebook from inside the course folder (lessons/<lesson>/notebook)."
sys.path.insert(0, str(NW))
from brightcart.incidents import fresh, generate
from brightcart.sim import World

try:
    from dotenv import load_dotenv
    load_dotenv(NW.parent / ".env")
except ImportError:
    pass
KEY = os.environ.get("GROQ_API_KEY")

import httpx
from openai import BadRequestError, OpenAI
from openai.types.chat import ChatCompletion

client = OpenAI(api_key=KEY or "missing", base_url="https://api.groq.com/openai/v1", max_retries=8)
MODEL = "openai/gpt-oss-20b"
CACHE = HERE / "cache"


def chat(sample=0, **body):
    """One model call with record/replay: a cached reply is replayed for free, a new one calls Groq once and is saved.
    A 400 (malformed tool call) is cached too and raised again as BadRequestError, like the real client does."""
    h = hashlib.sha256((json.dumps(body, sort_keys=True, default=str) + f"#{sample}").encode()).hexdigest()[:20]
    f = CACHE / f"{h}.json"
    if f.exists():
        saved = json.loads(f.read_text(encoding="utf-8"))
        if "error_400" in saved:
            raise BadRequestError(saved["error_400"], body=None,
                                  response=httpx.Response(400, request=httpx.Request("POST", str(client.base_url))))
        saved.pop("service_tier", None)            # Groq sends "on_demand", which the SDK type does not list
        return ChatCompletion.model_validate(saved)
    if not KEY:
        raise RuntimeError("This reply is not in ./cache and GROQ_API_KEY is not set. Get the free key (lesson 0.2), "
                           "put GROQ_API_KEY=... in the course's .env, and re-run the setup cell.")
    CACHE.mkdir(exist_ok=True)
    try:
        r = client.chat.completions.create(**body)
    except BadRequestError as e:
        f.write_text(json.dumps({"error_400": str(e)}), encoding="utf-8")
        raise
    f.write_text(r.model_dump_json(), encoding="utf-8")
    return r


def check(n, test):
    """Run an exercise check; print a clear line instead of stopping the notebook."""
    try:
        test()
        print(f"✓ Exercise {n} passed")
    except Exception as e:
        print(f"✗ Exercise {n} not yet: {type(e).__name__}: {str(e) or 'an assert failed, see the check cell'}")


print("nightwatch found:", NW.name, "| GROQ_API_KEY:", "set" if KEY else
      "NOT set -> cells marked 'calls the API' replay from ./cache; a new question needs the free key (lesson 0.2)")

nightwatch found: nightwatch | GROQ_API_KEY: set


## 1. Recall: one call, one result (lessons 2.1–2.2)

When the model asks for one tool, your code sends back one message with role `tool`, holding the result and the `tool_call_id` of the call it answers.

In [2]:
one = {"role": "tool", "tool_call_id": "fc_1d4c", "content": json.dumps(["04:13 ERROR checkout upstream payments returned 500"])}
print(one["role"], "answers", one["tool_call_id"])

tool answers fc_1d4c


## 2. Key words

| Word | Meaning |
|---|---|
| parallel tool calls | one assistant reply that asks for several tools at once |
| round trip / turn | one request to the model plus its reply |
| latency | the time you wait, from asking until the answer is ready |
| thread pool | a team of Python workers that run several functions at the same time |
| ID matching | each result carries the id of its call, so the order does not matter |

## 3. The idea: one reply, several calls

`tool_calls` is a **list**. Three rules for a batch:
1. Every id gets **exactly one** tool message back; none is skipped.
2. The results come right after that assistant message; the **id**, not the position, says which is which.
3. The calls don't depend on each other, so your code **may** run them at the same time.

### Hand trace (the video's run, Qwen's reply, last 8 characters of each id)

| # | role | content |
|---|---|---|
| 1 | system | "You are Nightwatch. Gather everything you need in as few turns as possible." |
| 2 | user | "Checkout error rate is 23%. Check checkout and payments status, payments logs and recent deploys." |
| 3 | assistant | 4 calls: `r9cb7cvb` service_status(checkout), `sp3z4xf8` service_status(payments), `2rd37hg5` read_logs(payments), `nv793n5g` recent_deploys() |
| 4 | tool `r9cb7cvb` | checkout v3.0.1, ... |
| 5 | tool `sp3z4xf8` | payments v1.3.5, ... |
| 6 | tool `2rd37hg5` | `04:13 ERROR payments KeyError: 'currency' in serializer (v1.3.5)` ... |
| 7 | tool `nv793n5g` | deploy at 04:12, payments v1.3.5, by ci |

Four calls, four results: the list grows from 3 messages to 7. Drop message 7 and the key clue, the deploy, vanishes.

## 4. Write it yourself

Three tools over the Brightcart simulator. Each one sleeps 0.8 s, as if it talked to a remote system.

In [3]:
import time
from concurrent.futures import ThreadPoolExecutor

world = World(fresh(next(x for x in generate() if x.id == "upstream_deploy-0")))
QWEN = "qwen/qwen3.8-27b"


def fn(name, desc, props=None, req=None):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props or {}, "required": req or []}}}


TOOLS = [fn("service_status", "Error %, p95 latency, CPU and memory of one service.",
            {"service": {"type": "string"}}, ["service"]),
         fn("read_logs", "Last log lines of one service.", {"service": {"type": "string"}}, ["service"]),
         fn("recent_deploys", "Deploys in the last hour.")]
SLOW = 0.8


def run_tool(name, args):
    time.sleep(SLOW)
    if name == "service_status":
        return world.status(args["service"])
    if name == "read_logs":
        return world.logs(args["service"], 6)
    if name == "recent_deploys":
        return world.inc.deploys
    raise KeyError(name)


SYSTEM = "You are Nightwatch. Gather everything you need in as few turns as possible."
ASK = "Checkout error rate is 23%. Check checkout and payments status, payments logs and recent deploys."
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ASK}]
print([t["function"]["name"] for t in TOOLS])

['service_status', 'read_logs', 'recent_deploys']


### 4.1 How many tool calls in one reply? (3 tries each, temperature 0)
Video: gpt-oss-20b → `[1, 1, 1]`, Qwen → `[4, 4, 4]`.

In [4]:
# calls the API (replays from ./cache when the reply is saved)
replies = {}
for model in (MODEL, QWEN):
    counts = []
    for i in range(3):
        r = chat(sample=i, model=model, temperature=0, messages=msgs, tools=TOOLS)
        replies.setdefault(model, r)
        counts.append(len(r.choices[0].message.tool_calls or []))
    print(f"{model:22} -> {counts}")

m = replies[QWEN].choices[0].message
calls = m.tool_calls or []
print("\nqwen's reply asks for:")
for c in calls:
    print(f"  {c.id[-8:]}  {c.function.name}({c.function.arguments})")

openai/gpt-oss-20b     -> [1, 1, 1]
qwen/qwen3.8-27b       -> [4, 4, 1]

qwen's reply asks for:
  h5xydc3g  service_status({"service":"checkout"})
  j1gsbrwa  service_status({"service":"payments"})
  k38ethg4  read_logs({"service":"payments"})
  d8xy6mke  recent_deploys({})


In the run saved here Qwen batched all four on two tries out of three (`[4, 4, 1]`; the video's run got `[4, 4, 4]`), and gpt-oss never batched. Batching is a habit of the model, not a rule of the API, so your code must handle both.

### 4.2 One by one, then all at once

In [5]:
def run_one(c):
    return run_tool(c.function.name, json.loads(c.function.arguments or "{}"))


t0 = time.time()
seq = [run_one(c) for c in calls]
t_seq = time.time() - t0

t0 = time.time()
with ThreadPoolExecutor(len(calls) or 1) as ex:
    par = list(ex.map(run_one, calls))          # results come back in call order
t_par = time.time() - t0
print(f"each tool takes {SLOW} s | one by one: {t_seq:.1f} s | at once: {t_par:.1f} s | same results: {seq == par}")

each tool takes 0.8 s | one by one: 3.2 s | at once: 0.8 s | same results: True


Threads pay off here because the tools spend their time **waiting**, not computing.

### 4.3 Every call gets exactly one result, matched by id

In [6]:
assistant = {"role": "assistant", "content": m.content or "", "tool_calls": [c.model_dump() for c in calls]}
results = [{"role": "tool", "tool_call_id": c.id, "content": json.dumps(res, default=str)} for c, res in zip(calls, par)]
for t in results:
    print(f"  tool result for {t['tool_call_id'][-8:]}: {t['content'][:70]}")
print("messages:", len(msgs), "->", len(msgs + [assistant] + results))

  tool result for h5xydc3g: {"checkout": {"version": "v3.0.1", "replicas": 3, "cpu": 26, "mem": 31
  tool result for j1gsbrwa: {"payments": {"version": "v1.3.5", "replicas": 2, "cpu": 39, "mem": 53
  tool result for k38ethg4: ["04:13 ERROR payments KeyError: 'currency' in serializer (v1.3.5)", "
  tool result for d8xy6mke: [{"t": "04:12", "service": "payments", "version": "v1.3.5", "by": "ci"
messages: 2 -> 7


### 4.4 The whole loop: ask, run every call at once, send all results back, repeat
A malformed tool call comes back as a 400; the loop nudges and retries (lesson 2.4 explains it). Video: gpt-oss 6 turns / 5 calls / 7.5 s, Qwen 2 turns / 4 calls / 2.7 s. Times vary with the API's speed; replayed from the cache, only the tool sleeps count.

In [7]:
# calls the API (replays from ./cache when the reply is saved)
def loop(model, max_turns=8):
    m_ = list(msgs)
    turns, n_calls, t0, bad = 0, 0, time.time(), 0
    while turns < max_turns:
        try:
            r = chat(model=model, temperature=0, messages=m_, tools=TOOLS)
        except BadRequestError as e:
            bad += 1
            print(f"   [{model}] 400 on turn {turns + 1}: {str(e).split(' - ', 1)[-1][:70]} -> nudge and retry")
            if bad > 3:
                return turns, n_calls, time.time() - t0, "(gave up after 400s)"
            m_.append({"role": "user", "content": "Your last tool call had invalid JSON arguments. "
                                                  "Call one tool again with valid JSON."})
            continue
        msg = r.choices[0].message
        turns += 1
        if not msg.tool_calls:
            return turns, n_calls, time.time() - t0, msg.content
        m_.append({"role": "assistant", "content": msg.content or "", "tool_calls": [c.model_dump() for c in msg.tool_calls]})
        with ThreadPoolExecutor(len(msg.tool_calls)) as ex:
            res = list(ex.map(run_one, msg.tool_calls))
        n_calls += len(msg.tool_calls)
        for c, x in zip(msg.tool_calls, res):
            m_.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(x, default=str)})
    return turns, n_calls, time.time() - t0, "(gave up)"


for model in (MODEL, QWEN):
    turns, n, secs, ans = loop(model)
    print(f"{model:22} turns {turns}  tool calls {n}  time {secs:.1f} s")
    print(f"   answer: {(ans or '')[:220]}")

   [openai/gpt-oss-20b] 400 on turn 4: Failed to parse tool call arguments as JSON -> nudge and retry


openai/gpt-oss-20b     turns 6  tool calls 5  time 4.0 s
   answer: **Current System Snapshot**

| Service | Version | Replicas | CPU % | Mem % | Error % | p95 Latency ms | Restarts |
|---------|---------|----------|-------|-------|---------|----------------|----------|
| **checkout** | 


qwen/qwen3.8-27b       turns 2  tool calls 4  time 0.8 s
   answer: ## Findings

**Root cause: payments v1.3.5 (deployed 04:12 by CI) introduced a serializer bug.**

### Evidence
- **payments logs** show `ERROR KeyError: 'currency' in serializer (v1.3.5)` at 04:13 — one minute after the 


One tool per turn means more round trips, so it is slower; this measures time, not diagnosis quality. gpt-oss stays the course default, and the same loop handles both styles.

## 5. The library version + check

The standard library's `ThreadPoolExecutor.map` keeps call order. `asyncio` is the other common way to overlap waiting: `asyncio.to_thread` + `gather` gives the same results in about the same time.

In [8]:
import asyncio, sys
print("python", sys.version.split()[0])


async def run_all_async(cs):
    return await asyncio.gather(*(asyncio.to_thread(run_one, c) for c in cs))

t0 = time.time()
par_async = await run_all_async(calls)          # notebooks allow top-level await
t_async = time.time() - t0
print(f"asyncio.gather: {t_async:.1f} s | same results as the pool: {par_async == par}")

python 3.13.16


asyncio.gather: 0.8 s | same results as the pool: True


## 6. Where this shows up in agents: reads together, writes alone

When an alert fires, Nightwatch first gathers facts: status, logs, deploys. These are **reads**, so running them together is safe. Later it has **writes** (restart, rollback). Run two writes at once and nobody can say which happened first, or whether one undid the other. Rule: reads may run in parallel; writes run one at a time. Nightwatch's own loop (`agent_v0.run`) runs every call in order, one by one, which is always safe.

In [9]:
import agent_v0
reads = [t["function"]["name"] for t in agent_v0.TOOLS if t["function"]["name"] not in agent_v0.WRITE_TOOLS | {"finish"}]
print("safe to run together:", ", ".join(reads))
print("one at a time:       ", ", ".join(sorted(agent_v0.WRITE_TOOLS)))

safe to run together: service_status, read_logs, recent_deploys, list_flags, config_changes, queue_depth, query_db, read_runbook, check_health
one at a time:        clear_cache, restart_service, rollback_deploy, rotate_secret, scale_service, set_config, set_flag


## 7. Traps (each one runs)

**Trap 1: a missing result gives a silent wrong answer.** We send every result except the last (the deploys). Video: accepted, no error, and the model blamed checkout.

In [10]:
# calls the API (replays from ./cache when the reply is saved)
try:
    r4 = chat(model=QWEN, temperature=0, tools=TOOLS, messages=msgs + [assistant] + results[:-1])
    m4 = r4.choices[0].message
    print("accepted, no error. Model then:", "asks again for " + str([c.function.name for c in m4.tool_calls])
          if m4.tool_calls else "answers: " + (m4.content or "")[:300])
except BadRequestError as e:
    print("400:", str(e).split(" - ", 1)[-1][:200])

accepted, no error. Model then: answers: Here's the picture:

**Checkout (v3.0.1)** — the problem child
- Error rate **16.3%** (your 23% figure is consistent with this being the dominant contributor)
- p95 latency 98ms, CPU 26%, mem 31%, 0 restarts — resources look fine, so this is a logic/config issue, not capacity.

**Payments (v1.3.5)**


**Trap 2: pairing results by position after the order changed.** Results finish in any order; zip them with the wrong list and the logs get the deploys' label.

In [11]:
finished = list(reversed(list(zip([c.id for c in calls], par))))   # pretend they finished last-first
wrong = [{"tool_call_id": c.id, "content": json.dumps(res, default=str)} for c, (_, res) in zip(calls, finished)]
for c, t in zip(calls, wrong):
    print(f"{c.function.name:15} labelled with -> {t['content'][:50]}")
by_id = dict(finished)
print("fixed by id:", all(by_id[c.id] == res for c, res in zip(calls, par)))

service_status  labelled with -> [{"t": "04:12", "service": "payments", "version": 
service_status  labelled with -> ["04:13 ERROR payments KeyError: 'currency' in ser
read_logs       labelled with -> {"payments": {"version": "v1.3.5", "replicas": 2, 
recent_deploys  labelled with -> {"checkout": {"version": "v3.0.1", "replicas": 3, 
fixed by id: True


**Trap 3: writes in the thread pool.** Two restarts of the same service, at once: both start before either ends.

In [12]:
import threading
log, lock = [], threading.Lock()

def restart(service):
    with lock: log.append(f"start restart {service}")
    time.sleep(0.2)
    with lock: log.append(f"end restart {service}")

with ThreadPoolExecutor(2) as ex:
    list(ex.map(restart, ["payments", "payments"]))
print(log)
try:
    assert log[1].startswith("end"), "a second write started before the first one finished"
except AssertionError as e:
    print("AssertionError:", e)

['start restart payments', 'start restart payments', 'end restart payments', 'end restart payments']
AssertionError: a second write started before the first one finished


**Trap 4: assuming every model batches.** gpt-oss asked for one tool per reply (section 4.1). A loop that runs only `tool_calls[0]` silently drops Qwen's other three; one that expects four waits forever on gpt-oss.

In [13]:
for model, r in replies.items():
    tc = r.choices[0].message.tool_calls or []
    try:
        fourth = tc[3]
        print(model, "-> 4th call:", fourth.function.name)
    except IndexError as e:
        print(model, "-> IndexError:", e, f"(only {len(tc)} call)")

openai/gpt-oss-20b -> IndexError: list index out of range (only 1 call)
qwen/qwen3.8-27b -> 4th call: recent_deploys


## 8. Exercises

Check cells never call the API: they use Qwen's saved reply as plain dicts (`calls_d`).

### Exercise 1 (the video's exercise)
Add a fifth call, the **checkout logs** (`read_logs` with `{"service": "checkout"}`, id `"call_checkout_logs"`), to Qwen's four. Run all five in the pool, store the time in `t5` and the tool messages (one per call, in call order) in `results5`, and print the time and every result's id.

In [14]:
calls_d = [c.model_dump() for c in calls]      # the same 4 calls, as dicts


def run_one_d(c):
    return run_tool(c["function"]["name"], json.loads(c["function"]["arguments"] or "{}"))

In [15]:
calls5, t5, results5 = [], None, []
# YOUR CODE HERE

In [16]:
def _t1():
    assert len(calls5) == 5 and calls5[-1]["id"] == "call_checkout_logs", "4 of Qwen's calls + the checkout logs call"
    assert [t["tool_call_id"] for t in results5] == [c["id"] for c in calls5], "one tool message per call, same ids"
    assert json.loads(results5[-1]["content"]) == world.logs("checkout", 6), "the 5th result = the checkout logs"
    assert t5 is not None and t5 < 2 * SLOW, f"5 x {SLOW} s should overlap in the pool, took {t5}"
check(1, _t1)

✗ Exercise 1 not yet: AssertionError: 4 of Qwen's calls + the checkout logs call


### Exercise 2: match by id, whatever the finishing order
Write `tool_messages(calls, finished)`: `calls` are dicts, `finished` is a list of `(id, result)` in **any** order. Return one tool message per call, in call order. Raise `ValueError` naming the id if a call has no result.

In [17]:
def tool_messages(calls, finished):
    # YOUR CODE HERE
    return []

In [18]:
def _t2():
    fin = list(reversed([(c["id"], r) for c, r in zip(calls_d, par)]))
    out = tool_messages(calls_d, fin)
    assert [t["tool_call_id"] for t in out] == [c["id"] for c in calls_d]
    assert all(json.loads(t["content"]) == json.loads(json.dumps(r, default=str)) for t, r in zip(out, par)), "content by id"
    try:
        tool_messages(calls_d, fin[1:])
    except ValueError as e:
        assert calls_d[-1]["id"] in str(e), "name the missing id"
    else:
        raise AssertionError("a missing result must raise ValueError")
check(2, _t2)

✗ Exercise 2 not yet: AssertionError: an assert failed, see the check cell


### Exercise 3: catch the hook before you send
Write `missing_results(messages)`: return the ids of the **last** assistant message's tool calls that have no tool message after it (in call order). Then the trap-1 history would have been caught.

In [19]:
def missing_results(messages):
    # YOUR CODE HERE
    return None

In [20]:
def _t3():
    assert missing_results(msgs + [assistant] + results) == []
    assert missing_results(msgs + [assistant] + results[:-1]) == [calls[-1].id], "the deploys result is missing"
    assert missing_results(msgs + [assistant]) == [c.id for c in calls]
check(3, _t3)

✗ Exercise 3 not yet: AssertionError: an assert failed, see the check cell


### Exercise 4: reads together, writes alone
Write `plan(names, writes)`: split a list of tool names (in order) into batches. Consecutive reads share one batch; every write is a batch of its own; order is kept.

In [21]:
def plan(names, writes):
    # YOUR CODE HERE
    return []

In [22]:
def _t4():
    W = agent_v0.WRITE_TOOLS
    got = plan(["service_status", "read_logs", "rollback_deploy", "check_health", "restart_service", "restart_service"], W)
    assert got == [["service_status", "read_logs"], ["rollback_deploy"], ["check_health"],
                   ["restart_service"], ["restart_service"]], got
    assert plan([], W) == [] and plan(["read_logs"], W) == [["read_logs"]]
check(4, _t4)

✗ Exercise 4 not yet: AssertionError: []


## Solutions (try first!)

In [23]:
calls5 = calls_d + [{"id": "call_checkout_logs", "type": "function",
                     "function": {"name": "read_logs", "arguments": '{"service": "checkout"}'}}]
t0 = time.time()
with ThreadPoolExecutor(len(calls5)) as ex:
    res5 = list(ex.map(run_one_d, calls5))
t5 = time.time() - t0
results5 = [{"role": "tool", "tool_call_id": c["id"], "content": json.dumps(r, default=str)} for c, r in zip(calls5, res5)]
print(f"5 calls at once: {t5:.1f} s")
for t in results5:
    print("  result for", t["tool_call_id"][-8:])

5 calls at once: 0.8 s
  result for h5xydc3g
  result for j1gsbrwa
  result for k38ethg4
  result for d8xy6mke
  result for out_logs


In [24]:
def tool_messages(calls, finished):
    by_id = dict(finished)
    out = []
    for c in calls:
        if c["id"] not in by_id:
            raise ValueError(f"no result for call {c['id']}")
        out.append({"role": "tool", "tool_call_id": c["id"], "content": json.dumps(by_id[c["id"]], default=str)})
    return out

In [25]:
def missing_results(messages):
    last = max(i for i, x in enumerate(messages) if x["role"] == "assistant")
    answered = {x["tool_call_id"] for x in messages[last + 1:] if x["role"] == "tool"}
    return [c["id"] for c in messages[last].get("tool_calls") or [] if c["id"] not in answered]

In [26]:
def plan(names, writes):
    batches = []
    for n in names:
        if n in writes:
            batches.append([n])
        elif batches and batches[-1][0] not in writes:
            batches[-1].append(n)
        else:
            batches.append([n])
    return batches